In [1]:
"""
exp04 Step 01: Trend Exit Exploration (EXPLORATION ON 2005-2014 ONLY; context, PROTOCOL.md §9)

The 9 candidates of PROTOCOL.md §5 (sell when the 15:58 close has been more than x below the 200-session average for n
consecutive decisions, buy back when it is more than x above it; x in {0, 3%, 5%}, n in {1, 5, 10}) are replayed as ONE
continuous path over 2005-01-03 -> 2014-12-31 (start invested, no forced buy-back), against buy-and-hold over the same
sessions. The data is cut at so.config.EXPLORATION_DATA_CUTOFF_DATE_STR (2015-03-18): nothing of the walk-forward period
is read.

What it shows: how the benchmark behaves through 2008 (a slow bear market, the case the rule is built for) and 2010-2011.
Caveat: 2008 is known to everyone (hindsight-prone); the result does not change the grid or the selection (pre-registered).
Outputs: store04_experiments/exp04_trend_exit/step01_exploration_data/. Trials: 9 (stage "exploration").
"""
None

In [2]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp04_trend_exit import config as exp_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT EXIT AND RE-ENTRY SIMULATION FUNCTIONS
from so.core.reentry_simulation import get_buy_and_hold_result_dict
# IMPORT THE CONTINUOUS REPLAY FUNCTIONS
from so.core.continuous_replay import simulate_continuous_replay_dict, get_episode_scorecard_pdf, get_replay_summary_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp04_trend_exit.rules import get_rule_candidate_list, get_rule_builder_func
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [3]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step01_exploration_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "exploration")

Experiment:	exp04_trend_exit	Protocol:	1.0	Config Hash:	8f4c178319
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp04_trend_exit/step01_exploration_data/
ℹ️ No earlier 'exploration' entry of exp04_trend_exit in the trial log (config hash 8f4c178319).


In [4]:
# CALL FUNCTION TO READ THE MINUTE BARS UP TO THE EXPLORATION DATA CUTOFF (NOTHING FROM THE WALK-FORWARD PERIOD IS READ)
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=config.EXPLORATION_DATA_CUTOFF_DATE_STR)
# CALL FUNCTION TO CONVERT THE DATA TO ARRAYS AND BUILD THE DAILY TABLE
ohlcv_array_dict = get_ohlcv_array_dict(complete_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
# DEFINE THE EXPLORATION WINDOW
exploration_start_str, exploration_end_str = exp_config.EXPLORATION_REPLAY_START_DATE_STR, exp_config.EXPLORATION_REPLAY_END_DATE_STR
# DEFINE ONE PERIOD PER CALENDAR YEAR (FOR THE YEARS-WON COUNT)
session_date_arr = np.array(ohlcv_array_dict["session_date_list"])
window_date_arr = session_date_arr[(session_date_arr >= pd.Timestamp(exploration_start_str).date()) & (session_date_arr <= pd.Timestamp(exploration_end_str).date())]
year_period_pdf = pd.DataFrame([{"period_id": year, "period_start": min(d for d in window_date_arr if d.year == year), "period_end": max(d for d in window_date_arr if d.year == year)}
                                for year in sorted({d.year for d in window_date_arr})])
# DISPLAY INFORMATION
print(f"Data:\t{daily_pdf['date'].min()} -> {daily_pdf['date'].max()}\t({len(daily_pdf):,} sessions)\tReplay:\t{exploration_start_str} -> {exploration_end_str}\t({len(window_date_arr):,} sessions, {len(year_period_pdf)} years)")
print(f"First session with a 200-session average:\t{daily_pdf.loc[daily_pdf['ma200_dist_pct'].notna(), 'date'].iloc[0]}")

Data:	2005-01-03 -> 2015-03-18	(2,568 sessions)	Replay:	2005-01-03 -> 2014-12-31	(2,516 sessions, 10 years)
First session with a 200-session average:	2005-10-18


In [5]:
"""
The 9 candidates, each one continuous path (start invested, no forced buy-back), against buy-and-hold.
"""
# SIMULATE BUY AND HOLD
buy_hold_dict = get_buy_and_hold_result_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str)
print(f"Buy & hold:\ttotal return {buy_hold_dict['metric_dict']['total_return']:.2%} | Sharpe {buy_hold_dict['metric_dict']['sharpe_ratio']:.2f} | max drawdown {buy_hold_dict['metric_dict']['max_drawdown']:.2%}")
# DEFINE THE RULE BUILDER
rule_builder_func = get_rule_builder_func(daily_pdf)
# LISTS TO HOLD THE RESULTS
row_dict_list, scorecard_pdf_list = [], []
# ITERATE OVER THE CANDIDATES
for candidate_idx, candidate_dict in enumerate(get_rule_candidate_list()):
    # BUILD AND REPLAY THE CANDIDATE
    exit_signal_arr, reentry_func, reason_str = rule_builder_func(candidate_dict)
    simulation_dict = simulate_continuous_replay_dict(ohlcv_array_dict, exploration_start_str, exploration_end_str, exp_config.UNTOUCHED_START_DATE_STR,
                                                      volatility_arr_in=np.full(len(daily_pdf), np.nan), stop_k_in=None, reentry_func_in=reentry_func,
                                                      reentry_reason_str_in=reason_str, exit_signal_arr_in=exit_signal_arr, max_cash_sessions_in=exp_config.MAX_CASH_SESSIONS)
    # SUMMARIZE THE PATH (YEARS AS PERIODS)
    summary_dict = get_replay_summary_dict(simulation_dict["daily_equity_pdf"], buy_hold_dict["daily_equity_pdf"], year_period_pdf, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
    scorecard_pdf = get_episode_scorecard_pdf(simulation_dict, ohlcv_array_dict)
    scorecard_pdf_list.append(scorecard_pdf.assign(candidate_idx=candidate_idx, **candidate_dict))
    # STORE THE ROW
    row_dict_list.append({"candidate_idx": candidate_idx, **candidate_dict, "total_return": summary_dict["strategy_total_return"], "buy_hold_total_return": summary_dict["buy_hold_total_return"],
                          "excess_total_return": summary_dict["excess_total_return"], "annualized_return": summary_dict["strategy_annualized_return"],
                          "sharpe_ratio": summary_dict["strategy_sharpe_ratio"], "max_drawdown": summary_dict["strategy_max_drawdown"], "in_market_pct": summary_dict["in_market_pct"],
                          "exit_count": simulation_dict["metric_dict"]["signal_exit_count"], "mean_cash_sessions": simulation_dict["metric_dict"]["mean_cash_sessions"],
                          "bought_back_lower_pct": simulation_dict["metric_dict"]["positive_episode_pct"],
                          "mean_log_share_gain": float(scorecard_pdf["log_share_gain"].mean()) if len(scorecard_pdf) else np.nan,
                          "years_won": summary_dict["period_windows_beating_buy_hold"], "year_count": summary_dict["period_window_count"],
                          "annualized_log_excess": summary_dict["log_excess_annualized_log_excess"], "log_excess_ci_low": summary_dict["log_excess_ci_low"], "log_excess_ci_high": summary_dict["log_excess_ci_high"]})
# CONVERT THE RESULTS
exploration_pdf = pd.DataFrame(row_dict_list)
exploration_scorecard_pdf = pd.concat(scorecard_pdf_list, ignore_index=True)
# DISPLAY THE EXCESS RETURN GRID (BUFFER x CONFIRMATION)
display(exploration_pdf.pivot(index="buffer", columns="confirmation", values="excess_total_return").round(4))
# PREVIEW DATAFRAME
exploration_pdf.round(4)

Buy & hold:	total return 69.02% | Sharpe 0.36 | max drawdown -56.44%


confirmation,1,5,10
buffer,,,
0.00,-0.1862,-0.0119,0.1026
0.03,0.3142,0.1927,0.3836
0.05,0.3516,0.0844,0.4783


,candidate_idx,buffer,confirmation,total_return,buy_hold_total_return,excess_total_return,annualized_return,sharpe_ratio,max_drawdown,in_market_pct,exit_count,mean_cash_sessions,bought_back_lower_pct,mean_log_share_gain,years_won,year_count,annualized_log_excess,log_excess_ci_low,log_excess_ci_high
0,0,0.00,1,0.5039,0.6902,-0.1862,0.0417,0.4236,-0.2325,0.7555,33,18.6364,0.0303,-0.0035,1,10,-0.0117,-0.1032,0.1044
1,1,0.00,5,0.6783,0.6902,-0.0119,0.0532,0.4922,-0.2555,0.7944,16,32.3125,0.0625,-0.0004,1,10,-0.0007,-0.0859,0.1188
2,2,0.00,10,0.7927,0.6902,0.1026,0.0602,0.5252,-0.2475,0.8191,8,56.8750,0.1250,0.0074,4,10,0.0059,-0.0798,0.1191
3,3,0.03,1,1.0044,0.6902,0.3142,0.0721,0.6802,-0.2080,0.7535,4,155.0000,0.2500,0.0427,3,10,0.0171,-0.0638,0.1262
4,4,0.03,5,0.8829,0.6902,0.1927,0.0654,0.5806,-0.1981,0.7869,3,178.6667,0.3333,0.0361,3,10,0.0108,-0.0711,0.1232
5,5,0.03,10,1.0738,0.6902,0.3836,0.0758,0.6514,-0.1822,0.7928,3,173.6667,0.3333,0.0682,3,10,0.0205,-0.0574,0.1284
6,6,0.05,1,1.0418,0.6902,0.3516,0.0741,0.6830,-0.1573,0.7598,3,201.3333,0.3333,0.0631,1,10,0.0189,-0.0604,0.1306
7,7,0.05,5,0.7745,0.6902,0.0844,0.0591,0.5364,-0.2247,0.7773,3,186.6667,0.3333,0.0164,1,10,0.0049,-0.0781,0.1191
8,8,0.05,10,1.1685,0.6902,0.4783,0.0806,0.6800,-0.1774,0.8091,2,240.0000,0.5000,0.1246,4,10,0.0249,-0.0454,0.1300


In [6]:
"""
Episode scorecard of the textbook rule (x = 0, n = 1) and of the slowest candidate (x = 5%, n = 10).
S / R > 1 = bought back lower than sold; buy_hold_return_out = what holding earned while out.
"""
# DISPLAY THE TWO SCORECARDS
scorecard_col_list = ["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "buy_hold_return_out", "max_decline_out_pct", "cost_paid_pct", "open_at_end"]
for buffer, confirmation in [(0.0, 1), (0.05, 10)]:
    candidate_scorecard_pdf = exploration_scorecard_pdf[(exploration_scorecard_pdf["buffer"] == buffer) & (exploration_scorecard_pdf["confirmation"] == confirmation)]
    print(f"\nx = {buffer:.0%}, n = {confirmation}:\t{len(candidate_scorecard_pdf)} episodes, {int((candidate_scorecard_pdf['s_over_r'] > 1).sum())} bought back lower, "
          f"product of S/R {np.prod(candidate_scorecard_pdf['s_over_r']):.4f}")
    display(candidate_scorecard_pdf[scorecard_col_list].round(4))


x = 0%, n = 1:	33 episodes, 1 bought back lower, product of S/R 0.8921


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,buy_hold_return_out,max_decline_out_pct,cost_paid_pct,open_at_end
0,2005-10-18,117.90,2005-10-31,120.77,0.9762,9,0.0242,-0.0067,0.0003,False
1,2006-06-07,126.01,2006-06-08,126.24,0.9982,1,0.0017,-0.0240,0.0002,False
2,2006-06-09,125.55,2006-06-29,127.22,0.9869,14,0.0131,-0.0256,0.0002,False
3,2006-07-12,125.91,2006-07-25,126.98,0.9916,9,0.0083,-0.0280,0.0002,False
4,2006-07-27,126.34,2006-07-28,127.83,0.9883,1,0.0116,-0.0001,0.0002,False
5,2006-08-09,126.71,2006-08-10,127.28,0.9955,1,0.0043,-0.0035,0.0002,False
6,2006-08-11,126.84,2006-08-15,128.71,0.9855,2,0.0146,-0.0002,0.0002,False
7,2007-08-03,143.40,2007-08-06,146.71,0.9774,1,0.0229,-0.0061,0.0002,False
8,2007-08-14,143.05,2007-08-22,146.66,0.9754,6,0.0251,-0.0424,0.0002,False
9,2007-08-28,143.59,2007-08-29,146.66,0.9791,1,0.0212,-0.0001,0.0002,False



x = 5%, n = 10:	2 episodes, 1 bought back lower, product of S/R 1.2830


,exit_date,exit_fill_price,reentry_date,reentry_fill_price,s_over_r,sessions_out,buy_hold_return_out,max_decline_out_pct,cost_paid_pct,open_at_end
73,2008-01-25,133.00,2009-07-15,93.41,1.4238,370,-0.2978,-0.4955,0.0003,False
74,2011-08-17,119.57,2012-01-25,132.69,0.9011,110,0.1096,-0.1016,0.0002,False


In [7]:
# SAVE THE RESULTS
write_csv_file_to_path(exploration_pdf, f"{output_path_str}exploration_candidate_data.csv", "W")
write_csv_file_to_path(exploration_scorecard_pdf, f"{output_path_str}exploration_scorecard_data.csv", "W")

ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp04_trend_exit/step01_exploration_data/exploration_candidate_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp04_trend_exit/step01_exploration_data/exploration_candidate_data.csv' Has Been Saved!
ℹ️ Saving In Overwrite Mode.	(💾 In Local File Storage)
⚠️ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp04_trend_exit/step01_exploration_data/exploration_scorecard_data.csv' Not Found!	(💾 In Local File Storage)
✅ File:	'C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/exp04_trend_exit/step01_exploration_data/exploration_scorecard_data.csv' Has Been Saved!


In [8]:
# LOG THE 9 CANDIDATES AS EXPLORATION TRIALS (ONE WRITE)
exploration_trial_pdf = log_trial_pdf(exp_config, "exploration", exploration_pdf.assign(window=f"{exploration_start_str}->{exploration_end_str}"), ["buffer", "confirmation", "window"],
                                      [col for col in exploration_pdf.columns if col not in ["candidate_idx", "buffer", "confirmation"]], False,
                                      note_str_in="exp04 step01 exploration (2005-2014 continuous replay, context only)")
print(f"Trials logged:\t{len(exploration_trial_pdf)}")

Trials logged:	9


In [9]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)

,experiment,stage,trial_count,config_count,test_touched_count,first_logged_ts,last_logged_ts
0,exp01_minute_entry,signal_check,2,2,0,2026-10-05T15:17:38,2026-10-06T12:46:46
1,exp01_minute_entry,summary,1,1,0,2026-10-05T16:16:30,2026-10-05T16:16:30
2,exp01_minute_entry,validation,540,1,0,2026-10-05T15:36:53,2026-10-05T16:16:18
3,exp02_stop_reentry,exploration,14,1,0,2026-10-05T14:47:50,2026-10-05T14:47:50
4,exp02_stop_reentry,signal_check,1,1,0,2026-10-05T14:48:41,2026-10-05T14:48:41
5,exp02_stop_reentry,summary,1,1,0,2026-10-05T14:50:29,2026-10-05T14:50:29
6,exp02_stop_reentry,validation,792,1,0,2026-10-05T14:49:18,2026-10-05T14:49:48
7,exp03_ath_exit,exploration,15,1,0,2026-10-05T14:55:04,2026-10-05T14:55:04
8,exp03_ath_exit,summary,1,1,0,2026-10-05T14:56:49,2026-10-05T14:56:49
9,exp03_ath_exit,validation,660,1,0,2026-10-05T14:55:55,2026-10-05T14:56:12
